# Center Loss: does the update rule matter? -- Colab runner

One question, measured on FER2013. The class centres can be updated two ways:

- **`gradient`** -- centres are learnable parameters stepped by an SGD optimiser
  on the combined objective. This is what most public implementations do, and
  what the rest of this project does. The coefficient enters the centre
  gradient and the displacement is normalised by the mini-batch size.
- **`wen`** -- Algorithm 1 of the paper: a dedicated update normalised per class
  by `1 + n_j`, applied with its own rate, independent of the coefficient.

FER2013 is heavily imbalanced (7,035 *happy* against 366 *disgust*), which is
what makes the two rules separable. On the balanced room dataset they could not
be told apart.

Run the cells top to bottom on a **T4 GPU** runtime. Cell 6 does everything.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code and install dependencies

`fer_wen/` is fully standalone -- it imports nothing from the other experiment
folders -- but it needs the same third-party packages, so the shared
requirements file is used.

In [ ]:
import os

REPO_DIR = '/content/room-classification'
BRANCH   = 'main'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/SilviuBR24/room-classification.git {REPO_DIR}
!cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}

!pip install -q -r {REPO_DIR}/vit_s16_baseline/requirements.txt

%cd {REPO_DIR}
print('Code ready on branch:', BRANCH)
print('fer_wen present:', os.path.isdir(f'{REPO_DIR}/fer_wen'))

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Unpack the dataset

The same `fer2013_clean.zip` used by the previous experiment, copied to the VM's
local SSD first because reading many small files from the Drive mount is far
slower.

In [ ]:
import os, time, shutil, zipfile

DRIVE_ZIP = '/content/drive/MyDrive/Dissertation_Thesis/fer2013_clean.zip'
LOCAL_ZIP = '/content/fer2013_clean.zip'
DATA_ROOT = '/content/fer2013_clean'
EXPECTED  = {'train': 27182, 'val': 3398, 'eval': 3397}

if not os.path.isdir(os.path.join(DATA_ROOT, 'train')):
    if not os.path.exists(LOCAL_ZIP):
        t0 = time.time(); shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
        print(f'Copied zip Drive->local in {time.time()-t0:.0f}s')
    t1 = time.time()
    with zipfile.ZipFile(LOCAL_ZIP) as z:
        z.extractall('/content')
    print(f'Extracted in {time.time()-t1:.0f}s')
else:
    print(f'{DATA_ROOT} already present, skipping.')

ok = True
for split, n in EXPECTED.items():
    p = os.path.join(DATA_ROOT, split)
    counts = {c: len(os.listdir(os.path.join(p, c))) for c in sorted(os.listdir(p))}
    total = sum(counts.values()); ok &= total == n
    print(f'{split:6s}: {total:6d} (expected {n})')
    print(f'        {counts}')

assert ok, 'Image counts do not match the published FER2013 split.'
print('\nDataset verified.')

## 5. What to expect, and how to read it

Two things differ between the modes, not one. Both follow from the same design
choice, but they are separate effects and this comparison cannot separate them:

1. **Per-class normalisation** -- dividing by `1 + n_j` instead of by the batch
   size.
2. **Overall rate** -- the gradient mode's effective centre rate is
   `eta * lambda = 0.5 * 0.0005 = 0.00025`; Algorithm 1 applies `alpha = 0.5`
   directly, roughly two thousand times faster.

So a difference in the results shows that the two implementations behave
differently. It does **not** isolate the per-class normalisation. Saying so
plainly is part of reporting this honestly.

The `gradient` variant runs first as a **control**: this folder has its own
training loop, so that run must reproduce what the shared loop already produced
(test accuracy 0.6807). The runner prints that check.

## 6. Run both variants

Roughly 35 minutes each on a T4, so about 70 minutes total. Safe to interrupt:
a summary row is written after each variant, and re-running skips whatever
already finished.

In [ ]:
import os, sys, yaml, subprocess

PROJECT   = '/content/room-classification/fer_wen'
DATA_ROOT = '/content/fer2013_clean'
RUNS_DIR  = '/content/drive/MyDrive/Dissertation_Thesis/dissertation_runs'
os.chdir(PROJECT)

cfg = yaml.safe_load(open('config_wen.yaml'))
cfg['data']['train_dir'] = DATA_ROOT + '/train'
cfg['data']['val_dir']   = DATA_ROOT + '/val'
cfg['data']['eval_dir']  = DATA_ROOT + '/eval'
cfg['paths']['output_root']    = RUNS_DIR
cfg['training']['num_workers'] = os.cpu_count()
yaml.safe_dump(cfg, open('config_wen_colab.yaml', 'w'), sort_keys=False)

print('resolution :', cfg['model']['image_size'])
print('epochs     :', cfg['training']['epochs'],
      '| batch', cfg['training']['batch_size'],
      '| workers', cfg['training']['num_workers'])
print('lambda     :', cfg['training']['center_loss_weight'])
print('centre rate:', cfg['training']['center_loss_lr'])

env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
p = subprocess.Popen([sys.executable, '-u', 'compare_wen.py',
                      '--config', 'config_wen_colab.yaml'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
for line in p.stdout:
    sys.stdout.write(line.decode('utf-8', 'replace')); sys.stdout.flush()
print('exit code:', p.wait())

## 7. Centre alignment, side by side

The direct measurement. For each class, how closely the learned centre tracks
the empirical centroid of that class, against how many training images the
class has.

Under the gradient rule these are strongly correlated -- on the earlier
`fer_baseline` run, r = +0.843, with *disgust* at 0.248 against *happy* at
0.990. If Algorithm 1's per-class normalisation does what it is meant to, that
correlation should weaken.

In [ ]:
import glob, os, csv
import numpy as np

RUNS_DIR = '/content/drive/MyDrive/Dissertation_Thesis/dissertation_runs'

def alignment(run_glob):
    runs = sorted(glob.glob(f'{RUNS_DIR}/{run_glob}'))
    if not runs:
        return None
    ev = sorted(glob.glob(f'{runs[-1]}/outputs/eval_*/centre_alignment.csv'))
    if not ev:
        return None
    with open(ev[-1]) as fh:
        return list(csv.DictReader(fh))

g = alignment('*_fer_wen_gradient')
w = alignment('*_fer_wen_algorithm1')
assert g and w, 'run cell 6 first'

print(f"{'class':<10}{'train imgs':>12}{'gradient':>11}{'Algorithm 1':>13}")
print('-' * 46)
for rg, rw in zip(g, w):
    print(f"{rg['class']:<10}{int(rg['train_images']):>12,}"
          f"{float(rg['cosine_centre_vs_centroid']):>11.3f}"
          f"{float(rw['cosine_centre_vs_centroid']):>13.3f}")
print('-' * 46)

n  = np.array([int(r['train_images']) for r in g], float)
cg = np.array([float(r['cosine_centre_vs_centroid']) for r in g])
cw = np.array([float(r['cosine_centre_vs_centroid']) for r in w])
print(f"\ncorrelation with class frequency:")
print(f"   gradient    r = {np.corrcoef(n, cg)[0,1]:+.3f}")
print(f"   Algorithm 1 r = {np.corrcoef(n, cw)[0,1]:+.3f}")
print(f"\nmean alignment:  gradient {cg.mean():.3f}   Algorithm 1 {cw.mean():.3f}")

## 8. (Optional) Embedding geometry

Whether the two rules leave the representation in a different shape. Read the
scale-invariant columns: the ratio, the silhouette coefficient and the
Davies-Bouldin index. The absolute distances shrink under either rule, because
that is what the objective optimises.

In [ ]:
import glob, os, sys, subprocess

RUNS_DIR = '/content/drive/MyDrive/Dissertation_Thesis/dissertation_runs'
g = sorted(glob.glob(f'{RUNS_DIR}/*_fer_wen_gradient'))
w = sorted(glob.glob(f'{RUNS_DIR}/*_fer_wen_algorithm1'))
assert g and w, 'run cell 6 first'

env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
p = subprocess.Popen([sys.executable, '-u',
                      '../fer_baseline/plot_fer_embeddings.py',
                      '--runs', g[-1], w[-1],
                      '--labels', 'Centres by gradient', 'Centres by Algorithm 1',
                      '--out-dir', '../figuri/fer_wen_geometry'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
for line in p.stdout:
    sys.stdout.write(line.decode('utf-8', 'replace')); sys.stdout.flush()
p.wait()